# Directed, Result-Focused Analyses of Vitamin D Signatures (LINCS L1000)

This notebook presents **targeted, hypothesis-driven analyses** of transcriptomic responses to vitamin D and its analogs using LINCS L1000 Level-5 data. We focus on **clear questions, compact metrics, and publishable figures**.

**Objectives**
- Define and validate a **core Vitamin D (VDR) signature** across contexts.
- Quantify responses **by cell line** and **by analog** using a single **core score**.
- Test **dose–response** monotonicity (Spearman ρ) and estimate **potency** (slope of `core_score ~ log10(dose)` with 95% CI).
- Perform **pathway enrichment** (GSEA Preranked, Enrichr) to confirm biological themes (Hallmarks/Reactome).
- Assess **analog similarity** (correlation heatmaps; pooled vs. cell-balanced) and **robustness** of conclusions (alt core sizes).

**Inputs**
- Expression matrix: genes × signatures (LINCS L1000 Level-5, z-scores).
- Metadata: signature, cell line, analog (`cmap_name`), dose, time; gene annotations.

**Output (figures & tables)**
1. Core score distributions **by cell** and **by analog**.
2. **Dose–response**: Spearman ρ (with FDR) and **potency ranking** (median slopes + 95% CI).
3. **Enrichment dot-plots** (Hallmarks/Reactome) per cell and per analog; top UP/DOWN tables.
4. **Analog similarity** heatmaps (pooled vs. balanced).
5. **Robustness** checks (alt core vs. original).

**Notebook structure**
1. Setup & integrity checks (short).
2. Core genes and **core score** definition.
3. Core score **by cell** and **by analog** (plots).
4. **Dose–response** (ρ and slopes with CI).
5. **Pathway enrichment** (GSEA/Enrichr summaries).
6. **Analog similarity** and **robustness**.
7. Key takeaways.

> All code cells are modular and short; each section ends with a brief interpretation.

---

## 1. Setup & integrity checks

### 1.1. Scientific Stack & Plotting Setup

In [ ]:
# Core scientific stack
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Statistics and modeling
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_ind
from scipy.spatial.distance import pdist, squareform
from skbio.stats.distance import DistanceMatrix, permanova
from sklearn.decomposition import PCA

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning utilities (for scaling or decomposition if needed)
from sklearn.preprocessing import StandardScaler

# Configure plotting aesthetics
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("viridis")
%matplotlib inline

### 1.2. Data Setup

We begin by loading all project data tables into memory:  
- **Expression matrix** (genes × signatures).  
- **Signature metadata** (perturbation, dose, cell line, etc.).  
- **Compound metadata** (compound-level annotations).  
- **Cell line metadata** (cell type, lineage, disease).  
- **Gene metadata** (landmark vs. inferred, gene symbols).  

Having all tables available ensures that downstream analyses can seamlessly combine expression values with their biological and experimental context.

In [ ]:
# Define data directory and file paths
DATA_DIR = "../data/exports"

PATHS = {
    "exp":   f"{DATA_DIR}/expression_matrix_clean.parquet",   # expression matrix
    "sig":   f"{DATA_DIR}/signature_metadata_clean.csv",      # signature metadata
    "comp":  f"{DATA_DIR}/subset_compounds_meta.csv",         # compounds
    "cells": f"{DATA_DIR}/subset_cell_lines_meta.csv",        # cell lines
    "genes": f"{DATA_DIR}/subset_genes_meta.csv",             # genes
}

# Load all tables into memory
exp_matrix = pd.read_parquet(PATHS["exp"])
metadata   = pd.read_csv(PATHS["sig"])
compounds  = pd.read_csv(PATHS["comp"])
cell_lines = pd.read_csv(PATHS["cells"])
gene_info  = pd.read_csv(PATHS["genes"])

# Quick overview of dimensions
print(f"Expression matrix: {exp_matrix.shape[0]} genes × {exp_matrix.shape[1]} signatures")
print(f"Metadata rows:     {len(metadata)}")
print(f"Compounds:         {len(compounds)}")
print(f"Cell lines:        {len(cell_lines)}")
print(f"Genes:             {len(gene_info)}")

### Data Setup Conclusion

All data tables were successfully loaded:  
- Expression matrix with 12,328 genes × 258 signatures  
- 258 metadata entries  
- 12 compounds, 5 cell lines, and 12,328 genes  

The dataset is ready for downstream analyses.

---

### 1.3. Integrity Gatekeeper

Before performing directed analyses, we run a minimal integrity check to ensure that the expression matrix and metadata are fully aligned and free of basic issues.  
This step verifies:  
- Consistent signature identifiers across tables  
- No missing values or zero-variance features  
- Dose information available and usable  

In [ ]:
def minimal_gatekeeper(exp_matrix, metadata, compounds, cell_lines, gene_info, expected_n=None):
    # Identify signature ID column in metadata
    sig_id_col = next((c for c in ["sig_id", "distil_id", "signature_id", "id"] if c in metadata.columns), None)
    if sig_id_col is None:
        raise ValueError("Signature ID column not found in metadata.")
    
    # Expression–metadata alignment (same set and order of signatures)
    exp_cols = pd.Index(map(str, exp_matrix.columns))
    meta_ids = pd.Index(metadata[sig_id_col].astype(str))
    common = exp_cols.intersection(meta_ids)
    if expected_n is not None and len(common) != expected_n:
        raise AssertionError(f"Common signatures = {len(common)} (expected {expected_n}).")
    if len(exp_cols.difference(common)) or len(meta_ids.difference(common)):
        raise AssertionError("Expression and metadata do not contain the exact same signatures.")
    meta_aligned = metadata.set_index(sig_id_col).loc[exp_cols].reset_index().rename(columns={"index": sig_id_col})
    
    # Basic integrity: NA and zero variance
    if exp_matrix.isna().any().any():
        raise AssertionError("NA values found in expression matrix.")
    if (exp_matrix.var(axis=1) == 0).any():
        raise AssertionError("Zero-variance genes detected.")
    if (exp_matrix.var(axis=0) == 0).any():
        raise AssertionError("Zero-variance signatures detected.")
    
    # Dose usability: numeric and variable within groups
    dose_col = next((c for c in meta_aligned.columns if ("dose" in c.lower()) and ("unit" not in c.lower())), None)
    if dose_col is None:
        raise AssertionError("Numeric dose column not found in metadata.")
    meta_aligned["dose_value"] = pd.to_numeric(meta_aligned[dose_col], errors="coerce")
    if meta_aligned["dose_value"].isna().any():
        raise AssertionError("Non-numeric values in dose column.")
    group_keys = [k for k in ["pert_id", "cell_id"] if k in meta_aligned.columns]
    if not group_keys:
        raise AssertionError("Missing grouping keys (pert_id/cell_id).")
    var_by_group = meta_aligned.groupby(group_keys)["dose_value"].agg(lambda x: float(np.var(x, ddof=1)) if x.notna().any() else 0.0)
    if (var_by_group == 0).all():
        raise AssertionError("No within-group dose variation; dose–response analyses are not feasible.")
    
    # Referential checks against lookup tables (lightweight)
    if "pert_id" in meta_aligned.columns and "pert_id" in compounds.columns:
        missing_comp = set(meta_aligned["pert_id"]) - set(compounds["pert_id"])
        if missing_comp:
            raise AssertionError(f"Missing compound keys in 'compounds': {len(missing_comp)}.")
    if "cell_id" in meta_aligned.columns and "cell_id" in cell_lines.columns:
        missing_cells = set(meta_aligned["cell_id"]) - set(cell_lines["cell_id"])
        if missing_cells:
            raise AssertionError(f"Missing cell IDs in 'cell_lines': {len(missing_cells)}.")
    if "gene_id" in getattr(gene_info, "columns", []):
        missing_genes = set(map(str, exp_matrix.index)) - set(map(str, gene_info["gene_id"]))
        if missing_genes:
            raise AssertionError(f"Missing gene IDs in 'gene_info': {len(missing_genes)}.")
    
    summary = {
        "signatures": len(common),
        "genes": exp_matrix.shape[0],
        "dose_col": dose_col,
        "dose_min": float(meta_aligned["dose_value"].min()),
        "dose_max": float(meta_aligned["dose_value"].max()),
        "groups_with_variation": int((var_by_group > 0).sum()),
    }
    return meta_aligned, summary

# Run gatekeeper (expecting 258 signatures based on previous step)
metadata_aligned, gate_summary = minimal_gatekeeper(
    exp_matrix, metadata, compounds, cell_lines, gene_info, expected_n=258
)

print("Gatekeeper summary:", gate_summary)


### Integrity Gatekeeper Conclusion

- 258 signatures aligned with the expression matrix  
- 12,328 genes retained  
- Dose column detected: `pert_dose` (range ≈ 0.01 – 10 µM)  
- 35 compound–cell groups show within-group dose variation  

The dataset passes all minimal integrity checks and is suitable for dose–response analyses.

---

## 2. Utilities & parameters
### 2.1 Utilities (helpers used across sections)

This cell centralizes small, reusable helpers so later code stays short and readable:

- **ID↔symbol mapping**
  - `build_symbol_map(...)` → `Series` gene_id→gene_symbol (string keys)
  - `map_symbols_or_ids(...)` → maps a list of gene_ids to symbols (fallback to gene_id)
- **Ranked vectors & gene lists**
  - `make_preranked(series, sym_map)` → GSEA-Preranked table (`gene, score`) with dedup by |score|
- **Statistics**
  - `spearman_by_group(df, group_cols, x, y)` → Spearman ρ and p per group
  - `fit_slope_ols(df, x, y)` → OLS slope of `y ~ x` (via `np.polyfit`)
  - `add_fdr(df, p_col)` → Benjamini–Hochberg FDR column
  - `bootstrap_ci(values)` → 95% bootstrap CI for the median
- **Plot/output**
  - `savefig(fig, name, folder, dpi)` → save figure with consistent settings

All helpers are **framework-agnostic** (no side effects, no file writes unless you call `savefig`).


In [ ]:
# === Utils (compact, no new imports) ===

def to_str_index(idx_like) -> pd.Index:
    """Return a string Index from any index-like object."""
    return pd.Index(idx_like).astype(str)

def build_symbol_map(gene_info: pd.DataFrame,
                     symbol_cols=("gene_symbol","pr_gene_symbol","symbol"),
                     id_col="gene_id") -> pd.Series:
    """
    Build a Series mapping gene_id(str) -> gene_symbol (may contain NaN).
    Picks the first available symbol column in `symbol_cols`.
    """
    sym_col = next((c for c in symbol_cols if c in gene_info.columns), None)
    if sym_col is None or id_col not in gene_info.columns:
        # return empty mapping with no crash downstream
        return pd.Series(dtype=object)
    gi = gene_info[[id_col, sym_col]].drop_duplicates(subset=[id_col]).copy()
    gi[id_col] = gi[id_col].astype(str)
    gi = gi.set_index(id_col)[sym_col]
    gi.index = gi.index.astype(str)
    return gi

def map_symbols_or_ids(ids, sym_map: pd.Series):
    """
    Map a list/Index of gene_ids to symbols; fallback to the gene_id when missing/blank.
    Returns a list[str].
    """
    ids_str = to_str_index(ids)
    s = sym_map.reindex(ids_str).astype(object) if isinstance(sym_map, pd.Series) else pd.Series(index=ids_str, dtype=object)
    na_mask = s.isna() | (s.astype(str).str.strip() == "") | (s.astype(str).str.lower().isin(["nan","none"]))
    s.loc[na_mask] = ids_str[na_mask]
    return s.astype(str).tolist()

def make_preranked(series: pd.Series, sym_map: pd.Series) -> pd.DataFrame:
    """
    Build a two-column DataFrame ('gene','score') for GSEA Preranked.
    - Input `series`: index = gene_id (any dtype), values = score (float).
    - Deduplicates by gene keeping the entry with largest |score|.
    - Sorted by score descending.
    """
    s = series.copy()
    s.index = to_str_index(s.index)
    symbols = map_symbols_or_ids(s.index, sym_map)
    df = pd.DataFrame({"gene": symbols, "score": s.values})
    # deduplicate on gene by max |score|
    df = df.iloc[df["score"].abs().sort_values(ascending=False).index]
    df = df.drop_duplicates(subset="gene", keep="first")
    return df.sort_values("score", ascending=False).reset_index(drop=True)

def fit_slope_ols(df: pd.DataFrame, x="log_dose", y="core_score") -> float:
    """
    Return OLS slope of y ~ x using np.polyfit (requires >=2 unique x).
    """
    xvals = df[x].values
    yvals = df[y].values
    # numpy polyfit returns [slope, intercept] for deg=1
    slope = float(np.polyfit(xvals, yvals, 1)[0])
    return slope

def spearman_by_group(df: pd.DataFrame, group_cols, x="log_dose", y="core_score",
                      min_n=4, min_unique=2) -> pd.DataFrame:
    """
    Compute Spearman rho and p-value per group (vectorized loop).
    Filters groups with n<min_n or unique(x)<min_unique.
    """
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    rows = []
    for keys, sub in df.dropna(subset=[x, y]).groupby(group_cols):
        if len(sub) >= min_n and sub[x].nunique() >= min_unique:
            rho, p = stats.spearmanr(sub[x], sub[y])
            rec = {c: k for c, k in zip(group_cols, (keys if isinstance(keys, tuple) else (keys,)))}
            rec.update({"n": len(sub), "rho": float(rho), "pval": float(p)})
            rows.append(rec)
    return pd.DataFrame(rows)

def add_fdr(df: pd.DataFrame, p_col="pval", out_col="fdr_bh") -> pd.DataFrame:
    """
    Add BH-FDR column to a DataFrame with p-values in `p_col`.
    Returns the same DataFrame (for chaining).
    """
    if df is None or df.empty or p_col not in df.columns:
        return df
    df[out_col] = multipletests(df[p_col].values, method="fdr_bh")[1]
    return df

def bootstrap_ci(values, B=4000, alpha=0.05, random_state=0):
    """
    Nonparametric bootstrap CI for the median.
    Returns (lo, hi).
    """
    rng = np.random.RandomState(random_state)
    vals = np.asarray(values, dtype=float)
    if len(vals) == 0:
        return np.nan, np.nan
    if len(vals) == 1:
        return float(vals[0]), float(vals[0])
    boots = np.empty(B, dtype=float)
    for i in range(B):
        sample = rng.choice(vals, size=len(vals), replace=True)
        boots[i] = np.median(sample)
    lo, hi = np.percentile(boots, [100*alpha/2, 100*(1-alpha/2)])
    return float(lo), float(hi)

def savefig(fig, name: str, folder="../results/figures", dpi=400, transparent=False, tight=True):
    """
    Save a matplotlib figure with consistent settings.
    Usage: savefig(plt.gcf(), "core_by_cell")
    """
    out_dir = Path(folder)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{name}.png"
    if tight:
        fig.savefig(path, dpi=dpi, bbox_inches="tight", transparent=transparent)
    else:
        fig.savefig(path, dpi=dpi, transparent=transparent)
    print(f"[saved] {path}")
    return path


### 2.2. Parameters (global knobs)

Centralized configuration to keep the notebook reproducible and readable.  
If you tweak thresholds (e.g., `N_TOP`, `VOTE_MIN` or core sizes), do it here.

In [ ]:
# --- Parameters (edit here if needed) ---
SEED = 0
N_TOP = 50       # window size for per-context top/bottom genes (vote-count)
VOTE_MIN = 2     # minimum contexts to call a gene 'consensus'
CORE_UP_N = 42   # default core UP size (from previous consensus)
CORE_DN_N = 35   # default core DOWN size

FIG_DIR = "../results/figures"
SAVE_FIGS = False      # <— único lugar
SAVE_TABLES = False    # opcional, ver punto 4

np.random.seed(SEED)

### Setup, utilities, and parameters — summary

**Status**
- Data tables loaded and aligned (Level-5 z-scores and metadata available).
- Integrity checks passed (gatekeeper summary already printed).
- Reusable helpers registered (`ID↔symbol`, preranked builder, stats, savefig).
- Global parameters set (e.g., `N_TOP`, `VOTE_MIN`, `CORE_UP_N`, `CORE_DN_N`, `SEED`).

**Reproducibility**
- Keep all changes to thresholds in the *Parameters* cell only.
- Re-run cells 1–3 if you modify paths, imports, or parameters.


---

## 3. Core genes & core score — setup

We first aggregate expression **by cell line** to obtain a per-cell mean profile (genes × cells), and build a robust **ID→symbol** lookup.  
This prepares the inputs for:
- per-cell rankings,
- cross-cell consensus of genes, and
- the **core score** computation in the next step.

### 3.1 Per-cell mean profile & symbol map

**Goal.** Collapse individual signatures into a **genes × cells** matrix by averaging Level-5 z-scores **within each cell line**. Build a robust **gene_id → gene_symbol** lookup for readable outputs downstream.

**Procedure.**
1) Align signatures with metadata and **group by `cell_id`**.  
2) Compute the **per-cell mean** z-score for every gene (Level-5 already encodes treated vs control).  
3) Attach `gene_id` (as string) and map **symbols** where available; **fallback** to `gene_id` if missing.

**Why this matters.**  
The per-cell matrix provides a clean, comparable baseline to:
- rank genes **within each cell** (for consensus/core),
- construct **GSEA preranked** vectors and **Enrichr** lists,
- compute the **core score** later on.

**Outputs.**
- `res_cell`: DataFrame with columns `[gene_id, gene_symbol, <one column per cell>]`.  
- `sym_map_cell`: `Series` mapping `gene_id(str) → gene_symbol`.

**Notes.** No filtering is applied here; averaging reduces within-cell noise without changing directionality.


In [ ]:
# Identify signature ID column (robust to different schemas)
SIG_COL = next(c for c in ["sig_id", "distil_id", "signature_id", "id"] if c in metadata_aligned.columns)

# Mean z-score per cell line (genes × cells)
# Note: Level-5 already encodes treated vs control; we average within each cell_id.
labels = metadata_aligned.set_index(SIG_COL)["cell_id"]
expr_by_cell = exp_matrix.T.groupby(labels).mean().T

# Build result table and attach IDs/symbols
res_cell = expr_by_cell.copy()
res_cell.insert(0, "gene_id", res_cell.index.astype(str))

# Symbol map (gene_id[str] -> gene_symbol), using the utils helper
sym_map_cell = build_symbol_map(gene_info)
# Attach gene_symbol (fallback handled later where needed)
res_cell.insert(1, "gene_symbol",
                pd.Series(map_symbols_or_ids(res_cell["gene_id"], sym_map_cell), index=res_cell.index))

# Quick QC
print(f"Per-cell matrix: {res_cell.shape[0]} genes × {res_cell.shape[1]-2} cells")
print("Cells:", ", ".join([c for c in res_cell.columns if c not in ["gene_id","gene_symbol"]]))
display(res_cell.head(5)[["gene_symbol","gene_id"] + [c for c in res_cell.columns if c not in ["gene_id","gene_symbol"]][:3]])

### 3.2 Cross-cell consensus & core definition

**Goal.** Identify a compact, cross-cell “core VDR” gene set that captures the most consistent responses to vitamin D.

**Procedure (per gene):**
1) For each **cell line**, rank genes by the **per-cell mean Level-5 z-score** (treated vs. control already encoded).
2) Take the **top/bottom `N_TOP`** genes per cell (UP/DOWN lists).
3) **Vote-count** across cells: how many cell lines does each gene appear in the top/bottom lists?
4) Break ties by the **global mean z-score** across cells.
5) Define the **core sets** as the top genes by votes (and tie-breaker) up to sizes **`CORE_UP_N`** and **`CORE_DN_N`**.

**Outputs.**
- `consensus_up`, `consensus_down` tables (gene_id, symbol, votes, global_mean_z).
- `core_up_ids`, `core_dn_ids` (final core gene IDs) for downstream **core score** and enrichment.

**Notes.**
- Sensitivity is controlled by **`N_TOP`** (window) and **`VOTE_MIN`** (minimum cells supporting a gene).  
- We expect UP core to reflect stress/anti-proliferative/anti-inflammatory tone; DOWN core to reflect cell-cycle/biogenesis attenuation.


In [ ]:
from collections import Counter

# --- Prepare per-cell rankings ---
cell_cols = [c for c in res_cell.columns if c not in ["gene_id", "gene_symbol"]]

# Dict: cell -> Series (index=gene_id[str], values=mean z), sorted desc
rank_by_cell = {
    cell: (res_cell.set_index("gene_id")[cell].rename(cell)
           .rename_axis("gene_id")
           .sort_values(ascending=False)
           .rename_axis(None))
    for cell in cell_cols
}

def top_sets(cell, n=N_TOP):
    s = rank_by_cell[cell]
    up_ids = set(to_str_index(s.head(n).index))
    dn_ids = set(to_str_index(s.tail(n).index))
    return up_ids, dn_ids

# --- Vote-count across cells ---
up_votes = Counter()
dn_votes = Counter()
for cell in cell_cols:
    up_ids, dn_ids = top_sets(cell, n=N_TOP)
    up_votes.update(up_ids)
    dn_votes.update(dn_ids)

# --- Global effect across cells (tie-breaker) ---
avg_effect = (
    res_cell.set_index("gene_id")[cell_cols]
            .mean(axis=1)
            .rename("global_mean_z")
)
avg_effect.index = to_str_index(avg_effect.index)
avg_effect = avg_effect.reset_index().rename(columns={"index": "gene_id"})

# --- Build consensus tables (apply VOTE_MIN, sort, attach symbols) ---
def build_consensus_table(counter: Counter, kind: str):
    items = [(str(gid), cnt) for gid, cnt in counter.items() if cnt >= VOTE_MIN]
    if not items:
        return pd.DataFrame(columns=["gene_id", f"votes_{kind}", "gene_symbol", "global_mean_z"])
    df = pd.DataFrame(items, columns=["gene_id", f"votes_{kind}"])
    # attach symbol
    df["gene_symbol"] = map_symbols_or_ids(df["gene_id"], sym_map_cell)
    # attach global mean effect
    df = df.merge(avg_effect, on="gene_id", how="left")
    # sort by votes (desc), then by effect (UP: desc, DOWN: asc)
    if kind == "up":
        df = df.sort_values([f"votes_{kind}", "global_mean_z"], ascending=[False, False])
    else:
        df = df.sort_values([f"votes_{kind}", "global_mean_z"], ascending=[False, True])
    df = df.reset_index(drop=True)
    return df

consensus_up   = build_consensus_table(up_votes, kind="up")
consensus_down = build_consensus_table(dn_votes, kind="down")

# --- Define core gene sets by size (fallback if fewer rows than requested) ---
def pick_core_ids(consensus_df: pd.DataFrame, size: int, votes_col: str) -> pd.Index:
    if consensus_df.empty:
        return pd.Index([], dtype=str)
    take = min(size, len(consensus_df))
    return to_str_index(consensus_df.head(take)["gene_id"])

core_up_ids = pick_core_ids(consensus_up,   CORE_UP_N, votes_col="votes_up")
core_dn_ids = pick_core_ids(consensus_down, CORE_DN_N, votes_col="votes_down")

# (Optional) tidy views for quick inspection
core_up_tbl = consensus_up.loc[consensus_up["gene_id"].isin(core_up_ids),
                               ["gene_symbol","gene_id","votes_up","global_mean_z"]]
core_dn_tbl = consensus_down.loc[consensus_down["gene_id"].isin(core_dn_ids),
                                 ["gene_symbol","gene_id","votes_down","global_mean_z"]]

# --- Console summary (short) ---
print(f"Consensus UP: {len(consensus_up)} genes (vote ≥ {VOTE_MIN}); core_UP size = {len(core_up_ids)} (target {CORE_UP_N})")
print(f"Consensus DOWN: {len(consensus_down)} genes (vote ≥ {VOTE_MIN}); core_DOWN size = {len(core_dn_ids)} (target {CORE_DN_N})")

print("\nCore UP (top 5):")
display(core_up_tbl.head(5))

print("\nCore DOWN (top 5):")
display(core_dn_tbl.head(5))


### 3.3 Core score computation & overview plots

**Goal.** Collapse the core gene sets into a single **core score** per signature:

**Core score**
$$
\overline{z}(\text{core UP}) - \overline{z}(\text{core DOWN})
$$

using Level-5 z-scores. We then join scores with metadata and visualize distributions **by cell line** and **by analog**.

**Outputs.**
- `core_score_meta`: one row per signature with `core_up_mean`, `core_dn_mean`, `core_score` + metadata.
- Two overview plots: box/strip by **cell** and by **analog** (ordered by median score).


In [ ]:
# 3.3 — Core score per signature + overview plots (robust to missing cmap_name)

# ---- Ensure we can join 'cmap_name' (map from compounds if needed) ----
if "cmap_name" not in metadata_aligned.columns:
    if ("pert_id" in metadata_aligned.columns) and ("pert_id" in compounds.columns) and ("cmap_name" in compounds.columns):
        metadata_aligned = metadata_aligned.merge(
            compounds[["pert_id", "cmap_name"]].drop_duplicates("pert_id"),
            on="pert_id", how="left"
        )

# ---- Compute core score from core gene IDs ----
genes_str = pd.Index(exp_matrix.index.astype(str))
i_up = genes_str.isin(pd.Index(core_up_ids).astype(str))
i_dn = genes_str.isin(pd.Index(core_dn_ids).astype(str))

n_up_in = int(i_up.sum()); n_dn_in = int(i_dn.sum())
print(f"Coverage in matrix — UP: {n_up_in} / {len(core_up_ids)}, DOWN: {n_dn_in} / {len(core_dn_ids)}")

core_up_mean = exp_matrix.loc[i_up].mean(axis=0)
core_dn_mean = exp_matrix.loc[i_dn].mean(axis=0)
core_score = core_up_mean - core_dn_mean

scores_df = pd.DataFrame({
    "sig_id": exp_matrix.columns.astype(str),
    "core_up_mean": core_up_mean.values,
    "core_dn_mean": core_dn_mean.values,
    "core_score": core_score.values,
})

# ---- Join with metadata (cell, analog, dose) ----
meta_cols = ["cell_id", "cmap_name", "dose_value", "log_dose", "dose_bin"]
keep_cols = [c for c in meta_cols if c in metadata_aligned.columns]
join_meta = metadata_aligned[[SIG_COL] + keep_cols].copy()
join_meta[SIG_COL] = join_meta[SIG_COL].astype(str)

core_score_meta = scores_df.merge(
    join_meta.rename(columns={SIG_COL: "sig_id"}),
    on="sig_id", how="left"
)

# Quick peek
display(core_score_meta.head())


# (a) by cell line
order_cells = (core_score_meta.groupby("cell_id")["core_score"]
               .median().sort_values(ascending=False).index)

plt.figure(figsize=(5, 3.2))
sns.boxplot(data=core_score_meta, x="cell_id", y="core_score",
            order=order_cells, showcaps=True, showfliers=False)
sns.stripplot(data=core_score_meta, x="cell_id", y="core_score",
              order=order_cells, size=3, alpha=0.55, jitter=0.25, color="k")
plt.title("Vitamin D core score by cell line"); plt.xlabel("Cell line"); plt.ylabel("Core score")
plt.tight_layout()
if SAVE_FIGS:
    savefig(plt.gcf(), "core_score_by_cell", folder=FIG_DIR)
plt.show()

# (b) by analog (only if cmap_name is available)
if "cmap_name" in core_score_meta.columns and core_score_meta["cmap_name"].notna().any():
    order_analogs = (core_score_meta.groupby("cmap_name")["core_score"]
                     .median().sort_values(ascending=False).index)

    plt.figure(figsize=(6.5, 3.2))
    sns.boxplot(data=core_score_meta, x="cmap_name", y="core_score",
                order=order_analogs, showcaps=True, showfliers=False)
    sns.stripplot(data=core_score_meta, x="cmap_name", y="core_score",
                  order=order_analogs, size=3, alpha=0.55, jitter=0.25, color="k")
    plt.title("Vitamin D core score by analog"); plt.xlabel("Analog"); plt.ylabel("Core score")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    if SAVE_FIGS:
        savefig(plt.gcf(), "core_score_by_analog", folder=FIG_DIR)
    plt.show()
else:
    print("[info] 'cmap_name' not present in metadata; skipping analog plot.")


### Section 3 — Summary (core genes & core score)

**What we built**
- A per-cell mean matrix (genes × cells) with a robust ID→symbol map.
- Cross-cell consensus via vote-count → **core sets:** **UP = 42**, **DOWN = 35** (vote ≥ 2).
- A single **core score** per signature:  
  \[
  \overline{z}(\text{core UP}) - \overline{z}(\text{core DOWN})
  \]
  merged with metadata (`cell_id`, `cmap_name`, dose).

**Key patterns**
- **By cell line:** strongest median core scores in **PC3**, followed by **MCF7**; **U2OS/HA1E** are moderate; **A549** shows broader dispersion.
- **By analog:** **paricalcitol** tends to show the highest median core score; the remaining analogs cluster close together.

**Biological reading (from the core)**
- **UP:** stress/mTORC1 brake (DDIT4), metabolic rewiring (PHGDH), lipid trafficking (NPC1), signaling/ECM (ADGRE5, TSKU).
- **DOWN:** cell-cycle/replication and biosynthetic load (KIF20A, PCNA, RPS4Y1, CCDC86, PRSS23).

**Artifacts to carry forward**
- `core_up_ids`, `core_dn_ids` (lists of gene IDs),  
- `core_score_meta` (per-signature scores + metadata),  
- parameters controlling robustness: `N_TOP`, `VOTE_MIN`, `CORE_UP_N`, `CORE_DN_N`.

---

## 4.1 Dose–response monotonicity (Spearman ρ)

**Goal.** Test whether higher **log10(dose)** is associated with a stronger **core score**.

**Method.**
- Compute **Spearman ρ** between `log_dose` and `core_score`.
- Summarize **by cell line**, **by analog**, and **analog×cell**.
- Adjust p-values with **Benjamini–Hochberg (FDR)**.

**Output.**
- Three compact tables with `n`, `rho`, `pval`, `FDR`.
- These guide which contexts show the clearest dose–response to vitamin D.

In [ ]:
# 4.1 — Spearman correlations: core_score ~ log_dose (by cell, by analog, and analog×cell)

df = core_score_meta.copy()

# Ensure log_dose is available
if "log_dose" not in df.columns:
    if "dose_value" in df.columns:
        df["log_dose"] = np.log10(pd.to_numeric(df["dose_value"], errors="coerce").clip(lower=1e-6))
    else:
        raise ValueError("Missing dose_value/log_dose in metadata.")

# Helper to pretty print
def _format_corr(df_corr, nrows=None):
    if df_corr is None or df_corr.empty:
        return df_corr
    out = df_corr.copy()
    for c in out.columns:
        if c not in ["n", "cell_id", "cmap_name"]:
            out[c] = out[c].astype(float)
    out["rho"] = out["rho"].round(3)
    out["pval"] = out["pval"].apply(lambda x: f"{x:.2e}")
    if "fdr_bh" in out.columns:
        out["fdr_bh"] = out["fdr_bh"].apply(lambda x: f"{x:.2e}")
    if nrows is not None:
        out = out.head(nrows)
    return out

# By cell line
res_cell = spearman_by_group(df, "cell_id", x="log_dose", y="core_score", min_n=6, min_unique=3)
res_cell = add_fdr(res_cell, "pval")
res_cell = res_cell.sort_values(["fdr_bh", "rho"], ascending=[True, False]).reset_index(drop=True)

print("Dose–core association (by cell):")
display(_format_corr(res_cell))

# By analog (cmap_name)
if "cmap_name" in df.columns:
    res_analog = spearman_by_group(df, "cmap_name", x="log_dose", y="core_score", min_n=6, min_unique=3)
    res_analog = add_fdr(res_analog, "pval")
    res_analog = res_analog.sort_values(["fdr_bh", "rho"], ascending=[True, False]).reset_index(drop=True)

    print("\nDose–core association (by analog):")
    display(_format_corr(res_analog))
else:
    res_analog = pd.DataFrame()
    print("\n[info] 'cmap_name' not present; skipping analog-level table.")

# Analog × cell (granular contexts)
if "cmap_name" in df.columns:
    res_axc = spearman_by_group(df, ["cmap_name", "cell_id"], x="log_dose", y="core_score", min_n=5, min_unique=3)
    res_axc = add_fdr(res_axc, "pval")
    res_axc = res_axc.sort_values(["fdr_bh", "rho"], ascending=[True, False]).reset_index(drop=True)

    print("\nDose–core association (analog × cell):")
    display(_format_corr(res_axc))
else:
    res_axc = pd.DataFrame()
    print("\n[info] 'cmap_name' not present; skipping analog×cell table.")

# Keep for later steps
dose_rho_by_cell = res_cell
dose_rho_by_analog = res_analog
dose_rho_by_axc = res_axc

### 4.1 Interpretation — dose–core monotonicity (Spearman ρ)

**By cell line.**  
- **MCF7 (ρ≈0.60, FDR≈2e-7)**, **A549 (ρ≈0.55, FDR≈3e-5)**, and **PC3 (ρ≈0.45, FDR≈7e-4)** show a **positive, significant** association between `log_dose` and `core_score`.  
- **U2OS (ρ≈0.33, FDR≈0.18)** is weaker (not FDR-significant).  
- **HA1E (ρ≈0.18, FDR≈0.20)** shows no clear monotonicity.

**By analog.**  
- Strongest and significant: **ercalcitriol (ρ≈0.69, FDR≈1.0e-4)**, **tacalcitol (ρ≈0.50, FDR≈0.018)**, **seocalcitol (ρ≈0.50, FDR≈0.018)**, **paricalcitol (ρ≈0.58, FDR≈0.025)**.  
- **calcitriol (ρ≈0.26, FDR≈0.023)** and **maxacalcitol (ρ≈0.36, FDR≈0.045)** are modest but significant.  
- **calcipotriol (ρ≈0.13, FDR≈0.505)** is not significant.

**Analog × cell.**  
- Very high correlations in specific contexts (often with small *n*): e.g., **ercalcitriol–PC3** and **paricalcitol–PC3 (ρ=1.0, FDR≈0)**; **calcitriol–MCF7 (ρ≈0.81, FDR≈1.3e-5)**; **ercalcitriol/tacalcitol/seocalcitol–A549 (ρ≈0.82–0.83, FDR≈0.016–0.017)**.  
- Treat extreme ρ with caution when sample size is small.

**Takeaway.** The **core score increases with dose** in most settings, especially in **MCF7, A549, and PC3**, and for **ercalcitriol, tacalcitol, seocalcitol, and paricalcitol**.

## 4.2 Dose–response potency (linear slope β vs. log10 dose)

**Goal.** Quantify how much the **core score** increases per **log10 unit of dose**.

**Method.**
- Fit **OLS** per group: `core_score ~ 1 + log_dose` (HC3 robust SEs).
- Summaries **by cell**, **by analog**, and **analog×cell**.
- Report **β (slope)**, 95% CI, *p*-value, and (adj.) R²; control FDR (BH).

**Output.**
- Three compact tables sorted by FDR.
- Two illustrative scatter+fit plots (best cell and best analog).

In [ ]:
# 4.2 — Linear potency slopes: core_score ~ log_dose (HC3)

# --- Ensure x is available
df = core_score_meta.copy()
if "log_dose" not in df.columns:
    if "dose_value" in df.columns:
        df["log_dose"] = np.log10(pd.to_numeric(df["dose_value"], errors="coerce").clip(lower=1e-6))
    else:
        raise ValueError("Missing dose_value/log_dose.")

def ols_by_group(data, group_cols, x="log_dose", y="core_score", min_n=6, min_unique=3):
    """
    Fit y ~ 1 + x per group (HC3 robust SEs). Returns one row per group with β, CI, p, R².
    """
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    rows = []
    for gvals, sub in data.dropna(subset=[x, y]).groupby(group_cols, dropna=False):
        if not isinstance(gvals, tuple):
            gvals = (gvals,)
        if len(sub) < min_n or sub[x].nunique() < min_unique or sub[x].var() <= 0:
            continue
        X = sm.add_constant(sub[[x]].astype(float))
        yv = sub[y].astype(float)
        fit = sm.OLS(yv, X).fit(cov_type="HC3")
        beta = fit.params[x]
        se = fit.bse[x]
        t  = fit.tvalues[x]
        p  = fit.pvalues[x]
        ci_low, ci_high = fit.conf_int().loc[x].tolist()
        rows.append({
            **dict(zip(group_cols, gvals)),
            "n": len(sub),
            "beta": float(beta),
            "se": float(se),
            "t": float(t),
            "pval": float(p),
            "ci_low": float(ci_low),
            "ci_high": float(ci_high),
            "r2": float(fit.rsquared),
            "r2_adj": float(fit.rsquared_adj),
        })
    return pd.DataFrame(rows)

# --- Run models
slopes_cell = ols_by_group(df, "cell_id", min_n=6, min_unique=3)
slopes_cell = add_fdr(slopes_cell, "pval").sort_values(["fdr_bh","beta"], ascending=[True,False]).reset_index(drop=True)

if "cmap_name" in df.columns:
    slopes_analog = ols_by_group(df, "cmap_name", min_n=6, min_unique=3)
    slopes_analog = add_fdr(slopes_analog, "pval").sort_values(["fdr_bh","beta"], ascending=[True,False]).reset_index(drop=True)
else:
    slopes_analog = pd.DataFrame()

if "cmap_name" in df.columns:
    slopes_axc = ols_by_group(df, ["cmap_name","cell_id"], min_n=5, min_unique=3)
    slopes_axc = add_fdr(slopes_axc, "pval").sort_values(["fdr_bh","beta"], ascending=[True,False]).reset_index(drop=True)
else:
    slopes_axc = pd.DataFrame()

print("Potency slopes — by cell (β per log10 dose):")
display(slopes_cell.head(10))

if not slopes_analog.empty:
    print("\nPotency slopes — by analog:")
    display(slopes_analog.head(10))
else:
    print("\n[info] 'cmap_name' not present; skipping analog-level slopes.")

if not slopes_axc.empty:
    print("\nPotency slopes — analog × cell:")
    display(slopes_axc.head(12))
else:
    print("\n[info] 'cmap_name' not present; skipping analog×cell slopes.")

# --- Illustrative scatterplots (best cell & best analog) ---
def _best_value(df_slopes, label_col):
    if df_slopes is None or df_slopes.empty: 
        return None
    return df_slopes.iloc[0][label_col]

best_cell = _best_value(slopes_cell, "cell_id")
best_analog = _best_value(slopes_analog, "cmap_name") if not slopes_analog.empty else None

# (a) Best cell
if best_cell is not None:
    sub = df[df["cell_id"] == best_cell].dropna(subset=["log_dose","core_score"])
    plt.figure(figsize=(4.8, 3.2))
    sns.regplot(data=sub, x="log_dose", y="core_score", ci=95, scatter_kws={"s":35, "alpha":0.7})
    b = slopes_cell.loc[slopes_cell["cell_id"]==best_cell, "beta"].iloc[0]
    p = slopes_cell.loc[slopes_cell["cell_id"]==best_cell, "pval"].iloc[0]
    plt.title(f"{best_cell}: core_score ~ log_dose (β={b:.2f}, p={p:.1e})")
    plt.xlabel("log10(dose)"); plt.ylabel("Core score")
    plt.tight_layout(); plt.show()

# (b) Best analog
if best_analog is not None:
    sub = df[df["cmap_name"] == best_analog].dropna(subset=["log_dose","core_score"])
    plt.figure(figsize=(4.8, 3.2))
    sns.regplot(data=sub, x="log_dose", y="core_score", ci=95, scatter_kws={"s":35, "alpha":0.7})
    b = slopes_analog.loc[slopes_analog["cmap_name"]==best_analog, "beta"].iloc[0]
    p = slopes_analog.loc[slopes_analog["cmap_name"]==best_analog, "pval"].iloc[0]
    plt.title(f"{best_analog}: core_score ~ log_dose (β={b:.2f}, p={p:.1e})")
    plt.xlabel("log10(dose)"); plt.ylabel("Core score")
    plt.tight_layout(); plt.show()

# Keep for later steps
slope_by_cell = slopes_cell
slope_by_analog = slopes_analog
slope_by_axc = slopes_axc


### 4.2 Interpretation — dose–response potency (β vs. log10 dose)

**What β means.** In `core_score ~ log10(dose)`, **β** is the increase in core score for a **10×** increase in dose (HC3 robust SEs).

**By cell line**
- **MCF7:** β ≈ **0.37** (95% CI ~ 0.23–0.51), **FDR ≈ 1e-6** → strongest, clean linear dose–response.
- **A549:** β ≈ **0.27** (0.12–0.41), **FDR ≈ 7e-4**.
- **PC3:** β ≈ **0.25** (0.11–0.39), **FDR ≈ 7e-4**.
- **HA1E:** β ≈ 0.07 (n.s.). **U2OS:** β ≈ 0.09 (n.s.).

**By analog**
- **Ercalcitriol:** β ≈ **0.53**, **FDR ≈ 1.2e-4** → highest potency.
- **Paricalcitol / Tacalcitol / Seocalcitol:** β ≈ 0.27–0.38, **FDR < 0.05** → clear positive slopes.
- **Calcitriol:** β ≈ 0.16, **FDR ≈ 0.002** → modest but significant.
- **Maxacalcitol, Calcipotriol:** not significant.

**Analog × cell (granular)**
- Strong, well-determined slopes in specific contexts, e.g.  
  **Paricalcitol–PC3** β ≈ **0.66** (**FDR ≈ 1e-6**),  
  **Calcitriol–MCF7** β ≈ **0.45** (**FDR ≈ 1e-4**),  
  **Tacalcitol–PC3/A549** β ≈ **0.47–0.52** (**FDR ≈ 1e-4–1e-3**),  
  **Ercalcitriol–HA1E** β ≈ **0.46** (**FDR ≈ 5e-4**).  
  Several other pairs are positive but not FDR-significant (often with small *n*).

**Takeaway.** The **potency** of the VDR-like response (core score) increases with dose in multiple settings, led by **MCF7/A549/PC3** and the analog **ercalcitriol**. Results are consistent with Spearman monotonicity but now provide an **interpretable scale** (Δcore per 10× dose) and **confidence intervals**.

*QC notes.* β and ρ align in signal; differences reflect **sample size** and **linearity**. Extreme estimates with **n ≈ 5–6** should be treated as indicative rather than definitive.


### 4.3 Graphical summary of potency slopes (forest plots)

**Goal.** Visualize the **dose–response potency** estimated in 4.2 as a compact, comparable summary.

**What is plotted.**
- Each point is the **slope β** from `core_score ~ log10(dose)` for a group (cell or analog).
- **Horizontal bars** show the **95% CI** around β (HC3 robust SEs).
- **Color** indicates FDR significance (**FDR < 0.05** highlighted).
- **Point size** scales with **sample size (n)** per group.

**How to read.**
- **β > 0** → core score increases when dose increases (per 10× step).
- **CI crossing 0** → not statistically different from zero.
- Larger points = more data; narrow CIs = more precise estimates.

**Inputs.** `slopes_cell`, `slopes_analog` from 4.2; helper `savefig`/`FIG_DIR` from utils.  
**Outputs.** Two forest plots (by **cell line** and by **analog**). Optional saving via `SAVE_FIGS=True`.


In [ ]:
# 4.3 — Graphical summary of potency slopes (forest plots with 95% CIs)

def _forest_slopes(df, group_col, title, fname=None):
    """
    Horizontal 'forest' plot of β with 95% CI.
    One dot per group; dot size ~ n; color marks FDR<0.05.
    """
    if df is None or df.empty:
        print(f"[info] No data to plot for {title}.")
        return

    d = df.copy().sort_values("beta", ascending=True)
    d["sig"] = d["fdr_bh"] < 0.05

    # marker size scaled by n
    n_min, n_max = d["n"].min(), d["n"].max()
    if n_max == n_min:
        sizes = np.full(len(d), 40.0)
    else:
        sizes = 30 + 90 * (d["n"] - n_min) / (n_max - n_min)

    # figure height adapts to number of rows
    h = max(2.8, 0.45 * len(d) + 1.2)
    plt.figure(figsize=(6.2, h))

    # baseline at 0
    plt.axvline(0, color="0.7", lw=1, zorder=0)

    # plot each row with its own color and x-error
    for i, row in enumerate(d.itertuples(index=False)):
        beta = row.beta
        lo, hi = row.ci_low, row.ci_high
        err_left, err_right = beta - lo, hi - beta
        color = "#4c78a8" if row.sig else "0.55"
        ms = sizes[i]
        plt.errorbar(
            beta, i, xerr=[[err_left], [err_right]],
            fmt="o", color=color, ecolor=color, elinewidth=1.2, capsize=3, markersize=ms**0.5
        )

    ylabels = d[group_col].astype(str) + " (n=" + d["n"].astype(str) + ")"
    plt.yticks(range(len(d)), ylabels)
    plt.xlabel("Slope β (Δ core score per 10× dose)")
    plt.title(title)
    plt.tight_layout()
    if SAVE_FIGS:
        savefig(plt.gcf(), fname or f"slopes_{group_col}", folder=FIG_DIR)
    plt.show()

# (a) Slopes by cell line
_forest_slopes(slopes_cell, "cell_id", "Dose–response potency by cell line", fname="slopes_by_cell")

# (b) Slopes by analog
if 'slopes_analog' in globals() and not slopes_analog.empty:
    _forest_slopes(slopes_analog, "cmap_name", "Dose–response potency by analog", fname="slopes_by_analog")
else:
    print("[info] Skipping analog plot (no slopes_analog).")


### Section 4 — Summary (dose–response)

**What we tested**
- **Monotonicity:** Spearman ρ between `log10(dose)` and `core_score` (per cell, per analog, and analog×cell) with BH FDR.
- **Potency:** linear slope **β** from `core_score ~ log10(dose)` using HC3 robust SEs + 95% CIs; summarized with forest plots.

**Key patterns**
- **By cell line:** clear dose–response in **MCF7**, **A549**, **PC3** (ρ and β significant); **U2OS** and **HA1E** are weak/non-significant.
- **By analog:** highest potency for **ercalcitriol**; **paricalcitol / tacalcitol / seocalcitol** also positive; **calcitriol** modest but significant; **maxacalcitol** and **calcipotriol** not significant.
- **Analog × cell:** several strong contexts (e.g., **paricalcitol–PC3**, **calcitriol–MCF7**, **tacalcitol–PC3/A549**, **ercalcitriol–HA1E**); treat extremes cautiously when *n* is small.

**Biological reading**
- The **VDR-like core program** strengthens with dose in specific cellular contexts and for particular analogs, consistent with a graded activation of stress/mTORC1 restraint, redox/NF-κB buffering, and reduced proliferation.

**Artifacts to carry forward**
- `core_score_meta` (per-signature scores + `log_dose`, metadata),
- `slopes_cell`, `slopes_analog`, `slopes_axc` (β, CI, *p*, FDR, R²),
- forest plots summarizing β **by cell** and **by analog**.

---
